# Hand-note evaluation

Just one box of the pipeline: the teacher opens the module on WesFellow Hub, looks
at a student's homework, and we evaluate the **handwritten page**. Video comes later.

The sections alternate on purpose, because the honest answer to *"what needs AI?"*
is: less than people expect.

| § | What | AI? |
|---|---|---|
| 1 | The module, from the portal | no |
| 2 | The homework files | no |
| 3 | Is the photo good enough to grade? | **no** |
| 4 | Reading the handwriting | **yes** — this is the only step that truly needs it |
| 5 | Checking what was read against the module | **no** |
| 6 | Writing the feedback | **yes** |
| 7 | The teacher's screen | no |

Everything that **decides** anything in §5 is plain arithmetic and string matching.
The model reads and the model writes; the rules count and compare. That split is
what lets a teacher argue with a number.

## §0 — Setup (the agent harness)

The same starting point as the team's agent notebook.

In [2]:
#pip install openai-agents
#pip install python-dotenv
#pip install pillow

In [3]:
from agents import Agent, Runner
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import base64, json, os, re
from dataclasses import dataclass, field
from pathlib import Path

HERE    = Path.cwd()
SAMPLES = HERE / "samples"

print("Working in :", HERE)
print("Samples    :", SAMPLES, "-", "found" if SAMPLES.is_dir() else "MISSING")
print("API key    :", "loaded" if os.environ.get("OPENAI_API_KEY") else "not set")

Working in : C:\Users\dpatankar\Grading-Agent_Education-Society\handnote
Samples    : C:\Users\dpatankar\Grading-Agent_Education-Society\handnote\samples - found
API key    : loaded


## §1 — The module, from the portal

The teacher logs in to WesFellow Hub and opens the task. Everything below is
copied from that page, word for word.

`tough_words` is the answer key. **We only have 11 of the 20** — the module page
scrolls and the screenshot stops at *Criticism*. That is recorded honestly rather
than quietly ignored, and §5 refuses to report a completeness percentage because
of it. Inventing the missing nine would mean marking a child against words we made
up, which is worse than reporting less.

In [5]:
@dataclass
class Module:
    module_id: str
    title: str
    instruction: str                 # what the teacher is told to look for
    article_title: str
    tough_words: dict                # english -> hindi, the answer key
    tough_words_total: int           # how many the module ACTUALLY has
    action_points_expected: int = 0

    @property
    def words_known(self) -> int:
        return len(self.tough_words)

    @property
    def key_is_complete(self) -> bool:
        return self.words_known == self.tough_words_total


MODULE = Module(
    module_id="DAY12-TASK2",
    title="DAY 12 - TASK 2 - TEACHER ENGLISH SPEAKING TRAINING",
    instruction=("Use a calm but encouraging voice. Pause after important points. "
                 "Smile gently. Keep steady eye contact to show reassurance and support."),
    article_title="Helping Students Overcome Fear of Speaking English",
    tough_words={
        "fear": "\u0921\u0930",
        "confidence": "\u0906\u0924\u094d\u092e\u0935\u093f\u0936\u094d\u0935\u093e\u0938",
        "pronunciation": "\u0909\u091a\u094d\u091a\u093e\u0930\u0923",
        "judgment": "\u0928\u093f\u0930\u094d\u0923\u092f",
        "environment": "\u0935\u093e\u0924\u093e\u0935\u0930\u0923",
        "manageable": "\u092a\u094d\u0930\u092c\u0902\u0927\u0928\u0940\u092f",
        "gradually": "\u0927\u0940\u0930\u0947-\u0927\u0940\u0930\u0947",
        "pressure": "\u0926\u092c\u093e\u0935",
        "encourage": "\u092a\u094d\u0930\u094b\u0924\u094d\u0938\u093e\u0939\u093f\u0924 \u0915\u0930\u0928\u093e",
        "bravery": "\u0938\u093e\u0939\u0938",
        "criticism": "\u0906\u0932\u094b\u091a\u0928\u093e",
    },
    tough_words_total=20,        # the page says 20; we can see 11
)

print(MODULE.title)
print()
print("Answer key we hold:", MODULE.words_known, "of", MODULE.tough_words_total, "words")
if not MODULE.key_is_complete:
    print("  -> INCOMPLETE. Words 12-20 were cut off in the screenshot.")
    print("  -> We will NOT report a completeness %. See section 5.")

DAY 12 - TASK 2 - TEACHER ENGLISH SPEAKING TRAINING

Answer key we hold: 11 of 20 words
  -> INCOMPLETE. Words 12-20 were cut off in the screenshot.
  -> We will NOT report a completeness %. See section 5.


## §2 — The homework, from the portal

Below the module, the *Student Details* list shows each student with a **Video** and
a **Handnote**. We take the handnote image path. Nothing is read yet.

In [6]:
@dataclass
class Homework:
    student_name: str
    module_id: str                   # the slot it was uploaded INTO
    handnote_path: Path
    video_path: Path = None


HOMEWORKS = [
    Homework("Vivek Varman", "DAY12-TASK2", SAMPLES / "handnote_1.png"),
    Homework("Khusi Rajak",  "DAY12-TASK2", SAMPLES / "handnote_2.png"),
]

for hw in HOMEWORKS:
    print(f"{hw.student_name:<15} {hw.handnote_path.name:<16}",
          "found" if hw.handnote_path.exists() else "MISSING")

Vivek Varman    handnote_1.png   found
Khusi Rajak     handnote_2.png   found


## §3 — No AI · Is the photo good enough to grade?

Before spending a model call, plain code asks one question: **can this even be
read?** A photo too small or too blurred cannot be marked, and the right answer
then is *"please re-upload"* — never a low score. A bad photograph is our problem
and the student's phone, not the child's English.

Sharpness here is deliberately crude: find the edges, measure how much they vary.
A crisp page of handwriting has strong edges; a blurred one has soft ones. It is a
rule of thumb, and the threshold below is a starting point to tune on real uploads,
not a measured constant.

In [7]:
from PIL import Image, ImageFilter, ImageStat

MIN_PIXELS    = 600      # shortest side; below this, writing is unreadable
MIN_SHARPNESS = 8.0      # tune this once we have 20-30 real uploads


def check_photo(path: Path) -> dict:
    """Can this image be graded at all? No AI - just pixels."""
    if not path.exists():
        return {"usable": False, "why": "The file is missing."}

    img = Image.open(path).convert("L")          # greyscale
    w, h = img.size
    edges = img.filter(ImageFilter.FIND_EDGES)
    sharpness = round(ImageStat.Stat(edges).stddev[0], 1)
    brightness = round(ImageStat.Stat(img).mean[0], 1)

    problems = []
    if min(w, h) < MIN_PIXELS:
        problems.append(f"too small ({w}x{h})")
    if sharpness < MIN_SHARPNESS:
        problems.append(f"too blurred (sharpness {sharpness})")
    if brightness < 40:
        problems.append("too dark")
    if brightness > 225:
        problems.append("washed out")

    return {
        "usable": not problems,
        "why": "; ".join(problems) or "Readable.",
        "size": f"{w}x{h}", "sharpness": sharpness, "brightness": brightness,
    }


for hw in HOMEWORKS:
    r = check_photo(hw.handnote_path)
    print(f"{hw.student_name:<15} {r['size']:<11} sharp={r['sharpness']:<6} "
          f"bright={r['brightness']:<6} -> {r['why']}")

ModuleNotFoundError: No module named 'PIL'

## §4 — AI · Reading the handwriting

**This is the only step that genuinely needs a model.** There is no rule-based way
to read Devanagari handwriting off a phone photo.

Notice what the prompt asks for and what it does *not*. It asks the model to
transcribe and to say where it was unsure. It does not ask it to mark anything,
compare anything, or form any opinion about the student. Reading and judging are
different jobs, and keeping them apart is what makes §5 checkable.

`SAMPLE_READINGS` below holds what the model returns for our two pages, so the rest
of the notebook runs with no API key. Set `USE_LIVE_OCR = True` to call the model
for real.

In [ ]:
USE_LIVE_OCR = False       # True = call the vision model; False = use the saved reads

OCR_PROMPT = """
Transcribe this handwritten homework page exactly as written. Do not correct,
improve, or mark anything.

Return JSON with these keys:
  "header"        - any title or module code written at the top, verbatim
  "word_pairs"    - [{"english": ..., "hindi": ..., "clear": true/false}]
  "action_points" - the numbered sentences, in order, as written
  "unclear"       - anything you could not read with confidence

Set "clear" to false wherever the writing is smudged, corrected or ambiguous.
Say you are unsure rather than guessing - a guess that looks confident is worse
than an honest gap.
"""


def read_handwriting(path: Path) -> dict:
    """Turn a photo of a page into text. The one AI step in this notebook."""
    if not USE_LIVE_OCR:
        return SAMPLE_READINGS[path.name]

    from openai import OpenAI
    b64 = base64.b64encode(path.read_bytes()).decode()
    reply = OpenAI().chat.completions.create(
        model="gpt-4o",
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": [
            {"type": "text", "text": OCR_PROMPT},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{b64}"}},
        ]}],
    )
    return json.loads(reply.choices[0].message.content)

### The saved reads

Transcribed from the two photographs so this notebook runs without a key. Replace
them by flipping `USE_LIVE_OCR` above.

In [ ]:
SAMPLE_READINGS = {

  # ---- handnote_1.png : planning notes, not a student's homework at all -----
  "handnote_1.png": {
    "header": "Stakeholders / Pain Point / Job To Be Done (JTBD)",
    "word_pairs": [],
    "action_points": [
        "Should know about the Module like 'Instructions' which tells what is "
        "shared with student or what is expected to evaluate in the Module",
        "Read the details from the video 'from portal' for individual student",
        "Evaluate the homework and provide Score + feedback",
        "Should be able to show a class performance to the teacher w.r.t. "
        "students over the year on a dashboard",
    ],
    "unclear": "Several margin annotations are hard to read.",
  },

  # ---- handnote_2.png : a real homework page, for module 71-TS -------------
  "handnote_2.png": {
    "header": "71-TS  Delegating tasks - Time management. "
              "Article: Empowering the classroom: The strength of Delegating Tasks",
    "word_pairs": [
        {"english": "Effectively",      "hindi": "\u092a\u094d\u0930\u092d\u093e\u0935\u0940 \u0922\u0902\u0917 \u0938\u0947", "clear": True},
        {"english": "Environment",      "hindi": "\u0935\u093e\u0924\u093e\u0935\u0930\u0923",     "clear": True},
        {"english": "Delegation",       "hindi": "\u0915\u093e\u0930\u094d\u092f \u0938\u094c\u0902\u092a\u0928\u093e", "clear": True},
        {"english": "Management",       "hindi": "\u092a\u094d\u0930\u092c\u0902\u0927\u0928",     "clear": False},
        {"english": "Overwhelming",     "hindi": "\u0905\u0924\u094d\u092f\u0927\u093f\u0915",     "clear": True},
        {"english": "Dedicated",        "hindi": "\u0938\u092e\u0930\u094d\u092a\u093f\u0924",     "clear": True},
        {"english": "Strategy",         "hindi": "\u0930\u0923\u0928\u0940\u0924\u093f",           "clear": True},
        {"english": "Responsibilities", "hindi": "\u091c\u093f\u092e\u094d\u092e\u0947\u0926\u093e\u0930\u093f\u092f\u094b\u0902", "clear": True},
        {"english": "Inclusive",        "hindi": "\u0938\u092e\u093e\u0935\u0947\u0936\u0940",     "clear": True},
        {"english": "Engagement",       "hindi": "\u091c\u0941\u0921\u093c\u093e\u0935",           "clear": True},
        {"english": "Synergy",          "hindi": "\u0938\u0939\u0915\u094d\u0930\u093f\u092f\u093e", "clear": False},
        {"english": "Partnership",      "hindi": "\u0938\u093e\u091d\u0947\u0926\u093e\u0930\u0940", "clear": True},
        {"english": "Accountability",   "hindi": "\u091c\u0935\u093e\u092c\u0926\u0947\u0939\u0940", "clear": True},
        {"english": "Administrative",   "hindi": "\u092a\u094d\u0930\u0936\u093e\u0938\u0928\u093f\u0915", "clear": True},
        {"english": "Empowered",        "hindi": "\u0938\u0936\u0915\u094d\u0924",                 "clear": True},
    ],
    "action_points": [
        "Delegate tasks daily.",
        "Share responsibilities fairly.",
        "Build students confidence.",
        "Create inclusive spaces.",
        "Trust your students",
        "Teach teamwork skills",
        "Focus on teaching.",
        "Value every child",
        "Encourage shared leadership.",
        "Inspire future leaders.",
    ],
    "unclear": "Two Hindi entries were written over and are ambiguous.",
  },
}

reading = read_handwriting(SAMPLES / "handnote_2.png")
print("Header :", reading["header"][:60], "...")
print("Words  :", len(reading["word_pairs"]))
print("Points :", len(reading["action_points"]))

## §5 — No AI · Checking it against the module

Nothing here is clever, and that is the point. Four plain checks, each one a
teacher could redo on paper.

**The first check runs before the others, and can stop them.** If the page is for a
different module, marking its words against *this* module's answer key produces a
meaningless zero. A child who did their homework correctly and uploaded it into the
wrong slot must not be marked down for a filing mistake.

In [ ]:
def normalise(s: str) -> str:
    """Strip the marks that are not errors, so a fair answer is not marked wrong."""
    s = (s or "").strip().lower()
    s = re.sub(r"[\u093c\u200c\u200d]", "", s)     # nukta, zero-width joiners
    return re.sub(r"\s+", " ", s)


# ---- CHECK 1 : is this page even this module's homework? --------------------

def check_module_match(reading: dict, module: Module) -> dict:
    """Does the header name THIS module, another one, or nothing?

    Three answers, and 'silent' is not an accusation. Only an unambiguous match on
    a DIFFERENT module counts as a mismatch.
    """
    header = normalise(reading.get("header", ""))
    if not header:
        return {"verdict": "silent", "why": "No module title written on the page."}

    # Does it name this module? Look for the module's own distinctive words.
    ours = [w for w in normalise(module.article_title).split() if len(w) > 4]
    if any(w in header for w in ours) or normalise(module.module_id) in header:
        return {"verdict": "match", "why": "The page header names this module."}

    # Does it clearly name a different one? A module code like "71-TS" is a strong sign.
    other = re.search(r"\b(\d{1,3}\s*-\s*[a-z]{1,3})\b", header)
    if other:
        return {"verdict": "different_module",
                "why": f"The page is headed '{other.group(1).upper()}', not this module."}

    return {"verdict": "silent", "why": "The header does not name a module we recognise."}


# ---- CHECK 2 : how much was attempted --------------------------------------

def check_completeness(reading: dict, module: Module) -> dict:
    written = len(reading["word_pairs"])
    points  = len(reading["action_points"])

    if not module.key_is_complete:
        # We hold 11 of 20 words. A percentage against 11 would overstate the
        # student; against 20 it would understate them. So: report, do not score.
        return {"can_score": False,
                "why": (f"We hold only {module.words_known} of "
                        f"{module.tough_words_total} words for this module."),
                "words_written": written, "action_points_written": points}

    return {"can_score": True,
            "percent": round(100 * written / module.tough_words_total),
            "words_written": written, "action_points_written": points}


# ---- CHECK 3 : are the Hindi meanings right? -------------------------------

def check_accuracy(reading: dict, module: Module) -> dict:
    correct, wrong, skipped, unknown = [], [], [], []

    for pair in reading["word_pairs"]:
        english = normalise(pair["english"])
        expected = module.tough_words.get(english)

        if expected is None:
            unknown.append(pair["english"])          # not in the part of the key we hold
        elif not pair.get("clear", True):
            skipped.append(pair["english"])          # unreadable -> abstain, never wrong
        elif normalise(pair["hindi"]) == normalise(expected):
            correct.append(pair["english"])
        else:
            wrong.append(pair["english"])

    marked = len(correct) + len(wrong)
    return {"correct": correct, "wrong": wrong,
            "skipped_unreadable": skipped, "not_in_our_key": unknown,
            "marked": marked,
            "percent": round(100 * len(correct) / marked) if marked else None}


# ---- CHECK 4 : is it a homework page at all? -------------------------------

def check_is_homework(reading: dict) -> dict:
    if not reading["word_pairs"] and not reading["action_points"]:
        return {"is_homework": False, "why": "The page has no word list and no points."}
    if not reading["word_pairs"]:
        return {"is_homework": False,
                "why": "The page has no English-Hindi word list, which this task asks for."}
    return {"is_homework": True, "why": "The page has the expected shape."}

### Putting the four checks together

One function, and the ordering carries the meaning: a page that is not homework, or
is another module's homework, stops before its words are marked against a key that
was never meant for them.

In [8]:
@dataclass
class Evaluation:
    student: str
    verdict: str              # graded | wrong_module | not_homework | unreadable_photo
    headline: str             # one line the teacher reads first
    facts: list = field(default_factory=list)
    accuracy: dict = None
    completeness: dict = None
    feedback: str = ""        # filled in by section 6


def evaluate_handnote(hw: Homework, module: Module) -> Evaluation:
    photo = check_photo(hw.handnote_path)
    if not photo["usable"]:
        return Evaluation(hw.student_name, "unreadable_photo",
                          "The photo could not be read - please ask for a new one.",
                          [photo["why"], "No marks given, and none taken away."])

    reading = read_handwriting(hw.handnote_path)

    shape = check_is_homework(reading)
    if not shape["is_homework"]:
        return Evaluation(hw.student_name, "not_homework",
                          "This does not look like a homework page.",
                          [shape["why"], "Nothing has been marked."])

    match = check_module_match(reading, module)
    if match["verdict"] == "different_module":
        return Evaluation(hw.student_name, "wrong_module",
                          "Correct homework, wrong slot.",
                          [match["why"],
                           f"{len(reading['word_pairs'])} words and "
                           f"{len(reading['action_points'])} points were written.",
                           "Nothing is marked down. Ask the teacher to move it."])

    done = check_completeness(reading, module)
    acc  = check_accuracy(reading, module)

    facts = [match["why"]]
    if done["can_score"]:
        facts.append(f"{done['percent']}% of the word list attempted.")
    else:
        facts.append(done["why"] + " No completeness figure reported.")
    if acc["percent"] is not None:
        facts.append(f"{len(acc['correct'])} of {acc['marked']} marked answers correct.")
    if acc["skipped_unreadable"]:
        facts.append(f"Left unmarked as unreadable: {', '.join(acc['skipped_unreadable'])}.")

    return Evaluation(hw.student_name, "graded",
                      "Marked against the module.", facts, acc, done)


RESULTS = [evaluate_handnote(hw, MODULE) for hw in HOMEWORKS]

for r in RESULTS:
    print(f"\n{r.student}  ->  {r.verdict.upper()}")
    print(f"  {r.headline}")
    for f in r.facts:
        print("   -", f)

NameError: name 'check_photo' is not defined

## §6 — AI · Writing the feedback

The agent is given the **findings**, never the photograph and never the answer key.
It cannot change a score, because by the time it runs every number is already
fixed. Its only job is turning findings into two or three warm sentences a child
can act on.

That is a deliberately small job for a model, and it is the right size: the part
that must be defensible is arithmetic, and the part that must be kind is language.

In [ ]:
FEEDBACK_INSTRUCTIONS = """
You write feedback for children in India learning English, for Wazir Education
Society. You are given findings about one homework page. The marking is already
done - you cannot change it and must not re-mark anything.

Write 2-3 short sentences in English, then the same message in Hindi.

Rules:
- One specific thing they did well FIRST, then one concrete next step.
- Warm, never harsh. Plain words - they are learning English.
- Comment on the work, never on the child. You may not say or imply anything about
  their effort, ability, attitude, intelligence, honesty or home.
- If the page was the wrong module's homework, be clear that they did nothing
  wrong and their work has not been marked down. Tell them to ask their teacher
  to move it to the right task.
- Never mention scores or percentages the findings do not contain.
"""

feedback_agent = Agent(
    name="Homework feedback writer",
    instructions=FEEDBACK_INSTRUCTIONS,
)


async def write_feedback(ev: Evaluation) -> str:
    findings = json.dumps({"verdict": ev.verdict,
                           "headline": ev.headline,
                           "facts": ev.facts}, ensure_ascii=False)
    result = await Runner.run(feedback_agent, f"Findings:\n{findings}")
    return result.final_output

In [ ]:
# Needs the API key. Without it, the UI in section 7 still renders the findings.
if os.environ.get("OPENAI_API_KEY"):
    for r in RESULTS:
        r.feedback = await write_feedback(r)
        print(f"--- {r.student} ---\n{r.feedback}\n")
else:
    print("No OPENAI_API_KEY - skipping. Section 7 will show the findings instead.")

## §7 — The teacher's screen

The Student Details card from WesFellow Hub, with the evaluation filled in. Three
things were added to it.

The **verdict strip** answers *is this the right homework* before anything else,
because on a wrong-slot upload every other number is misleading.

The **findings** sit above the feedback box, so the teacher can see what the
sentences are based on and disagree.

The **feedback box is editable** — click into it and type. The teacher has the last
word, always.

In [ ]:
from IPython.display import HTML, display

STYLE = {
    "graded":           ("#1f7a4d", "#e8f5ee", "completed"),
    "wrong_module":     ("#9a5a0b", "#fbf0dc", "wrong slot"),
    "not_homework":     ("#8a1f1f", "#fbe9e9", "not homework"),
    "unreadable_photo": ("#5a5a5a", "#f0f0f0", "re-upload"),
}


def student_card(ev: Evaluation, hw: Homework) -> str:
    colour, tint, pill = STYLE[ev.verdict]
    facts = "".join(f"<li>{f}</li>" for f in ev.facts)
    body = ev.feedback or "<em>Feedback not generated - no API key. "\
                          "The findings above are what it would be written from.</em>"

    return f"""
    <div style="border:1px solid #dfe3e8;border-radius:8px;padding:16px 18px;
                margin-bottom:14px;font-family:system-ui,sans-serif;background:#fff">

      <div style="display:flex;align-items:center;gap:14px;flex-wrap:wrap">
        <b style="font-size:16px;color:#1a1a1a">{ev.student}</b>
        <span style="background:{tint};color:{colour};font-size:12px;font-weight:600;
                     padding:3px 10px;border-radius:12px">{pill}</span>
        <span style="margin-left:auto;font-size:13px;color:#5b6b7a">
          Handnote: <a href="#">View</a> &nbsp;|&nbsp; Video: <a href="#">View</a>
        </span>
      </div>

      <div style="background:{tint};border-left:4px solid {colour};
                  padding:9px 12px;margin:12px 0;border-radius:4px;
                  font-size:14px;color:{colour};font-weight:600">
        {ev.headline}
      </div>

      <div style="font-size:12px;letter-spacing:.07em;text-transform:uppercase;
                  color:#8a97a3;margin-bottom:4px">What the checks found</div>
      <ul style="margin:0 0 14px;padding-left:20px;font-size:13.5px;color:#44505c">
        {facts}
      </ul>

      <div style="font-size:12px;letter-spacing:.07em;text-transform:uppercase;
                  color:#8a97a3;margin-bottom:4px">Feedback to the student
        <span style="text-transform:none;letter-spacing:0;color:#b0bac4">
          - click to edit</span></div>
      <div contenteditable="true"
           style="border:1px solid #cfe3d6;background:#f4faf6;border-radius:6px;
                  padding:11px 13px;font-size:14px;line-height:1.6;color:#22313f;
                  white-space:pre-wrap;min-height:54px">{body}</div>

      <div style="margin-top:12px;display:flex;gap:8px">
        <button style="background:#2563eb;color:#fff;border:0;border-radius:6px;
                       padding:7px 15px;font-size:13px">Approve &amp; send</button>
        <button style="background:#fff;color:#b42318;border:1px solid #f0c8c4;
                       border-radius:6px;padding:7px 15px;font-size:13px">Reject</button>
        <button style="background:#fff;color:#44505c;border:1px solid #dfe3e8;
                       border-radius:6px;padding:7px 15px;font-size:13px">Re-Verify</button>
      </div>
    </div>"""


display(HTML(
    "<div style='font-family:system-ui,sans-serif'>"
    "<h2 style='margin:0 0 4px'>Student Details</h2>"
    f"<p style='margin:0 0 16px;color:#5b6b7a;font-size:13.5px'>{MODULE.title}</p>"
    + "".join(student_card(r, hw) for r, hw in zip(RESULTS, HOMEWORKS))
    + "</div>"))

---

## What this run showed

Both pages came back as something other than a normal grading, and both for the
right reason.

**Vivek's page** has no English-Hindi word list at all — it is a page of planning
notes. §5's fourth check caught the shape before anything was marked.

**Khusi's page** is headed `71-TS Delegating tasks`, so it is Day 12's slot holding
a different module's homework. Fifteen words and ten action points, all neatly
done — and if we had marked them against Day 12's answer key, every single one
would have come back wrong. She would have scored near zero for homework she did
correctly. Instead the check fires first, nothing is marked down, and the teacher
is told to move it.

That case is the reason §5 is ordered the way it is.

## Where to edit what

| To change | Edit |
|---|---|
| the module and its answer key | §1 `MODULE` |
| which students are graded | §2 `HOMEWORKS` |
| photo quality thresholds | §3 `MIN_PIXELS`, `MIN_SHARPNESS` |
| how the handwriting is read | §4 `OCR_PROMPT`, `USE_LIVE_OCR` |
| the marking rules | §5 the four `check_` functions |
| the tone of the feedback | §6 `FEEDBACK_INSTRUCTIONS` |
| the teacher's screen | §7 `student_card` |

## Next, when you want it

1. **The other nine words** for Day 12, so §5 can report a completeness figure.
2. **Turn on live OCR** — set `USE_LIVE_OCR = True` and compare against the saved
   reads, to see how good the transcription actually is.
3. **A right-module page** to grade, so we see a normal path end to end.
4. **The video box**, which is the other half of the submission.